# Module 09 — Practical GPU Benchmarking Laboratory

This notebook is self-contained: it measures allocator state, memory bandwidth, GEMM throughput, projection shape efficiency, launch overhead, and host/device transfers on one T4.

## Lab protocol

Run on one NVIDIA T4. Warm up kernels, use CUDA events for device work, synchronize wall-clock measurements, and record shapes, dtypes, bytes, FLOPs, and memory state.

In [ ]:
# All benchmark helpers are defined here; this notebook does not import benchmark_lab.py.
import statistics
import time
import torch

assert torch.cuda.is_available(), 'Select a CUDA-enabled T4 runtime.'
device = torch.device('cuda:0')

def memory_report():
    free, total = torch.cuda.mem_get_info(device)
    return {'allocated_mib': torch.cuda.memory_allocated(device) / 2**20, 'reserved_mib': torch.cuda.memory_reserved(device) / 2**20, 'free_mib': free / 2**20, 'total_mib': total / 2**20}

def device_report():
    props = torch.cuda.get_device_properties(device)
    report = {'gpu': props.name, 'compute_capability': f'{props.major}.{props.minor}', 'pytorch': torch.__version__, 'cuda_runtime': torch.version.cuda}
    print(report); print('memory:', memory_report()); return report

def benchmark_gpu(operation, warmup=10, repeats=30, inner=1):
    for _ in range(warmup): operation()
    torch.cuda.synchronize(device); samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True); start.record()
        for _ in range(inner): operation()
        end.record(); end.synchronize(); samples.append(start.elapsed_time(end) / inner)
    return statistics.median(samples)

def benchmark_wall(operation, warmup=5, repeats=20):
    for _ in range(warmup): operation()
    torch.cuda.synchronize(device); samples = []
    for _ in range(repeats):
        torch.cuda.synchronize(device); started = time.perf_counter(); operation(); torch.cuda.synchronize(device)
        samples.append((time.perf_counter() - started) * 1000)
    return statistics.median(samples)

def allocation_experiment():
    before = memory_report(); tensor = torch.empty((1024, 1024), device=device); after = memory_report(); del tensor; return {'before': before, 'after': after}

def vector_add_experiment(vector_mib=128, warmup=10, repeats=30):
    n = vector_mib * 1024**2 // 4; a = torch.randn(n, device=device); b = torch.randn_like(a); c = torch.empty_like(a)
    ms = benchmark_gpu(lambda: torch.add(a, b, out=c), warmup, repeats); bytes_moved = 3 * n * 4
    return {'milliseconds': ms, 'effective_bandwidth_gbs': bytes_moved / (ms / 1000) / 1e9, 'arithmetic_intensity': 1 / 12}

def gemm_experiment(sizes, warmup=10, repeats=30):
    rows = []
    for size in sizes:
        a = torch.randn((size, size), device=device, dtype=torch.float16); b = torch.randn_like(a); c = torch.empty_like(a)
        ms = benchmark_gpu(lambda: torch.mm(a, b, out=c), warmup, repeats); flops = 2 * size**3
        rows.append({'size': size, 'milliseconds': ms, 'tflops': flops / (ms / 1000) / 1e12, 'arithmetic_intensity': flops / ((a.numel() + b.numel() + c.numel()) * 2)})
    return rows

def projection_experiment(batches, hidden_size=4096, warmup=10, repeats=30):
    weight = torch.randn((hidden_size, hidden_size), device=device, dtype=torch.float16); rows = []
    for batch in batches:
        x = torch.randn((batch, hidden_size), device=device, dtype=torch.float16); y = torch.empty_like(x)
        ms = benchmark_gpu(lambda: torch.mm(x, weight, out=y), warmup, repeats); flops = 2 * batch * hidden_size**2
        rows.append({'batch': batch, 'milliseconds': ms, 'tflops': flops / (ms / 1000) / 1e12})
    return rows

def launch_experiment(elements=1024, count=100, warmup=10, repeats=30):
    x = torch.ones(elements, device=device); y = torch.empty_like(x); xb = torch.ones((count, elements), device=device); yb = torch.empty_like(xb)
    many = lambda: [torch.mul(x, 2, out=y) for _ in range(count)]; one = lambda: torch.mul(xb, 2, out=yb)
    a = benchmark_wall(many, warmup, repeats); b = benchmark_wall(one, warmup, repeats); return {'many_launches_ms': a, 'one_launch_ms': b, 'ratio': a / b}

def transfer_experiment(transfer_mib=128, warmup=5, repeats=20):
    n = transfer_mib * 1024**2 // 4; cpu = torch.randn(n); gpu = torch.empty(n, device=device); host = torch.empty_like(cpu)
    h2d = benchmark_wall(lambda: gpu.copy_(cpu), warmup, repeats); d2h = benchmark_wall(lambda: host.copy_(gpu), warmup, repeats); bytes_moved = n * 4
    return {'h2d_ms': h2d, 'd2h_ms': d2h, 'h2d_gbs': bytes_moved / (h2d / 1000) / 1e9, 'd2h_gbs': bytes_moved / (d2h / 1000) / 1e9}

print('Device:', device); device_report()

## 1. GPU inspection and allocator state

In [ ]:
# Experiment 1: observe live allocation and allocator reservation.
before = memory_report()
allocation_tensor = torch.empty((1024, 1024), device=device)
after = memory_report()
print('before:', before)
print('after:', after)
del allocation_tensor

## 2. Memory bandwidth

FP32 vector addition has one FLOP and 12 useful bytes per element, so its arithmetic intensity is approximately 0.083 FLOPs/byte.

In [ ]:
# Experiment 2: measure useful data traffic for C = A + B.
vector_mib = 128
elements = vector_mib * 1024**2 // 4
a = torch.randn(elements, device=device, dtype=torch.float32)
b = torch.randn_like(a)
c = torch.empty_like(a)
milliseconds = benchmark_gpu(lambda: torch.add(a, b, out=c))
useful_bytes = 3 * elements * 4
vector = {'milliseconds': milliseconds, 'effective_bandwidth_gbs': useful_bytes / (milliseconds / 1000) / 1e9, 'arithmetic_intensity': 1 / 12}
print(vector)

## 3. GEMM throughput and projection shape efficiency

In [ ]:
# Experiment 3: measure square FP16 GEMM shapes and arithmetic throughput.
gemm = []
for size in [256, 512, 1024, 2048, 4096]:
    a_gemm = torch.randn((size, size), device=device, dtype=torch.float16)
    b_gemm = torch.randn_like(a_gemm)
    c_gemm = torch.empty_like(a_gemm)
    milliseconds = benchmark_gpu(lambda: torch.mm(a_gemm, b_gemm, out=c_gemm))
    flops = 2 * size**3
    gemm.append({'size': size, 'milliseconds': milliseconds, 'tflops': flops / (milliseconds / 1000) / 1e12})
print(gemm)

# Experiment 4: hold the weight matrix fixed while changing token rows.
hidden_size = 4096
weight = torch.randn((hidden_size, hidden_size), device=device, dtype=torch.float16)
projections = []
for batch in [1, 4, 16, 32, 64, 128, 256, 512]:
    x = torch.randn((batch, hidden_size), device=device, dtype=torch.float16)
    y = torch.empty_like(x)
    milliseconds = benchmark_gpu(lambda: torch.mm(x, weight, out=y))
    flops = 2 * batch * hidden_size**2
    projections.append({'batch': batch, 'milliseconds': milliseconds, 'tflops': flops / (milliseconds / 1000) / 1e12})
print(projections)

In [ ]:
# Plot projection throughput to expose shape-sensitive utilization.
import matplotlib.pyplot as plt
plt.plot([r['batch'] for r in projections], [r['tflops'] for r in projections], marker='o')
plt.xlabel('Input rows / token positions'); plt.ylabel('Achieved TFLOPS'); plt.grid(alpha=0.3); plt.show()

## 4. Launch overhead and CPU–GPU transfers

In [ ]:
# Experiment 5: compare many tiny launches with one batched launch.
count, elements = 100, 1024
x = torch.ones(elements, device=device); y = torch.empty_like(x)
xb = torch.ones((count, elements), device=device); yb = torch.empty_like(xb)
def many_launches():
    for _ in range(count): torch.mul(x, 2, out=y)
def one_launch(): torch.mul(xb, 2, out=yb)
many_ms = benchmark_wall(many_launches); one_ms = benchmark_wall(one_launch)
launch = {'many_launches_ms': many_ms, 'one_launch_ms': one_ms, 'ratio': many_ms / one_ms}

# Experiment 6: time host-to-device and device-to-host copies.
transfer_elements = 128 * 1024**2 // 4
cpu_tensor = torch.randn(transfer_elements)
gpu_tensor = torch.empty(transfer_elements, device=device)
cpu_destination = torch.empty_like(cpu_tensor)
h2d_ms = benchmark_wall(lambda: gpu_tensor.copy_(cpu_tensor))
d2h_ms = benchmark_wall(lambda: cpu_destination.copy_(gpu_tensor))
bytes_moved = transfer_elements * 4
transfers = {'h2d_ms': h2d_ms, 'd2h_ms': d2h_ms, 'h2d_gbs': bytes_moved / (h2d_ms / 1000) / 1e9, 'd2h_gbs': bytes_moved / (d2h_ms / 1000) / 1e9}
print('launch:', launch)
print('transfers:', transfers)

## Interpretation

Use the GPU report, allocator state, effective bandwidth, achieved TFLOPS, projection curve, launch ratio, and transfer rates to classify memory-bound, compute-oriented, and dispatch-sensitive behavior. These measurements form a reproducible baseline for later inference modules.